# 08 · Studi Kasus Curah Hujan BMKG — Bab 9

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 9: prediksi hujan stasiun — dua lintasan (regresi mm dan klasifikasi kategori), fitur stasiun + indeks iklim, baseline klimatologi, GRU/multivariate, walk-forward, verifikasi CSI/POD/FAR dengan threshold, dan interpretasi permutation importance.

## 1. Setup & Data Contoh

Data harian sintetik meniru musim (monsun) + indeks iklim. Ganti dengan data nyata BMKG + ERA5 untuk penggunaan sungguhan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

t = pd.date_range("2010-01-01", periods=12*365, freq="D")
doy = t.dayofyear.values
musim = np.maximum(18*np.sin(2*np.pi*(doy-15)/365.25), 0)
rmm = 0.8*np.sin(2*np.pi*np.arange(len(t))/45.0) + 0.3*np.sin(2*np.pi*np.arange(len(t))/90.0)
nino = 1.0*np.sin(2*np.pi*np.arange(len(t))/365.25*3)
hujan = np.maximum(np.random.gamma(1.3, 5, len(t)) + musim*(1+0.3*rmm) + 2*((nino>0.5)), 0)
hujan = np.where(np.random.rand(len(t))<0.5, 0.0, hujan)
suhu = 28 + 1.2*np.sin(2*np.pi*(doy-60)/365.25) + 0.8*np.random.randn(len(t))
df = pd.DataFrame({"r_hujan": hujan.round(1), "suhu": suhu.round(1), "rmm1": rmm, "nino34": nino}, index=t)
print(df.head())
print("Distribusi kategori lebat (>50):", int((df["r_hujan"]>50).sum()), "hari")

## 2. Feature Engineering (Tabel 9.1)

Lag, musiman sinus, indeks iklim.

In [ ]:
for lag in [1,2,3,7]:
    df[f"hujan_t{lag}"] = df["r_hujan"].shift(lag)
df["mus_sin"] = np.sin(2*np.pi*df.index.dayofyear/365.25)
df["mus_cos"] = np.cos(2*np.pi*df.index.dayofyear/365.25)
feat = [c for c in df.columns if c != "r_hujan"]
print("Fitur:", feat)

# target kategori (Tabel 9.2)
kategori = np.select([df["r_hujan"]<20, df["r_hujan"]<=50], [0,1], default=2)
df["kategori"] = kategori.astype(int)

## 3. Baseline Klimatologi "Cerdas"

Rata-rata per hari Julian dari data latih, lalu ulangi ke test.

In [ ]:
def mae(a,b): return float(np.mean(np.abs(a-b)))

df_ml = df.dropna().copy()
n = len(df_ml)
ntr = int(n*0.7); nva = int(n*0.15)

df_tr = df_ml.iloc[:ntr]
klim = df_tr.groupby(df_tr.index.dayofyear)["r_hujan"].mean()
base_test = klim.reindex(df_ml.index[ntr+nva:].dayofyear).fillna(0).values
y_test = df_ml["r_hujan"].values[ntr+nva:]
print("MAE klimatologi:", round(mae(y_test, base_test), 4))

## 4. Normalisasi (skala latih) + Windowing untuk GRU

In [ ]:
from sklearn.preprocessing import StandardScaler

w = 14
sc = StandardScaler().fit(df_ml[feat].iloc[:ntr])
Z = sc.transform(df_ml[feat])
Y = df_ml["r_hujan"].values
K = df_ml["kategori"].values

def buat_window(X, y, w=14):
    Xw, yw = [], []
    for i in range(len(X) - w):
        Xw.append(X[i:i+w]); yw.append(y[i+w])
    return np.array(Xw), np.array(yw)

Xw, Yw = buat_window(Z, Y, w)
_, Kw = buat_window(Z, K, w)
n2 = len(Xw)
ntr2, nva2 = int(n2*0.7), int(n2*0.15)
Xtr, Xva, Xte = Xw[:ntr2], Xw[ntr2:ntr2+nva2], Xw[ntr2+nva2:]
Ytr, Yva, Yte = Yw[:ntr2], Yw[ntr2:ntr2+nva2], Yw[ntr2+nva2:]
Ktr, Kva, Kte = Kw[:ntr2], Kw[ntr2:ntr2+nva2], Kw[ntr2+nva2:]
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

## 5. Regresi (GRU, transformasi log1p)

In [ ]:
bias = 1.0
m = tf.keras.Sequential([
    tf.keras.layers.GRU(16, input_shape=(w, Xtr.shape[2])),
    tf.keras.layers.Dense(1)])
m.compile(optimizer="adam", loss="mse", metrics=["mae"])
m.fit(Xtr, np.log1p(Ytr), validation_data=(Xva, np.log1p(Yva)),
      epochs=30, batch_size=32, verbose=0)
pred_log = m.predict(Xte, verbose=0).ravel()
pred = np.expm1(pred_log)
print("MAE GRU (mm, skala asli):", round(mae(Yte, pred), 4))

# klimatologi untuk test yang sejajar dengan pred (potong w)
klim_test = klim.reindex(df_ml.index[ntr+nva:].dayofyear[w:]).fillna(0).values
print("MAE klima (sejajar):", round(mae(Yte, klim_test), 4))

## 6. Klasifikasi Biner (Lebat vs Tidak) + Threshold (Kode 9.1)

In [ ]:
leb = (Kte >= 2).astype(int)   # lebat sebagai kelas positif
w_lap = {0: 1.0, 1: 8.0}

mc = tf.keras.Sequential([
    tf.keras.layers.GRU(16, input_shape=(w, Xtr.shape[2])),
    tf.keras.layers.Dense(1, activation="sigmoid")])
mc.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
mc.fit(Xtr, (Ktr>=2).astype(int), validation_data=(Xva, (Kva>=2).astype(int)),
      epochs=30, batch_size=32, class_weight=w_lap, verbose=0)
prob = mc.predict(Xte, verbose=0).ravel()

def verifikasi(y_true, prob, thresholds=[0.2, 0.4, 0.5, 0.6, 0.8]):
    baris = []
    for t in thresholds:
        yp = (prob>=t).astype(int)
        tp=((yp==1)&(y_true==1)).sum(); fp=((yp==1)&(y_true==0)).sum(); fn=((yp==0)&(y_true==1)).sum()
        pod=tp/(tp+fn) if tp+fn else 0; far=fp/(tp+fp) if tp+fp else 1
        csi=tp/(tp+fp+fn) if tp+fp+fn else 0
        baris.append((t, pod, far, csi))
    return pd.DataFrame(baris, columns=["threshold","POD","FAR","CSI"])

verifikasi(leb, prob).round(3)

## 7. Permutation Importance (Kode 9.2)

In [ ]:
def perm_imp(model, X, y, n=5):
    base = mae(y, model.predict(X, verbose=0).ravel())
    imp = {}
    for j in range(X.shape[2]):
        scores = []
        for _ in range(n):
            Xp = X.copy()
            rng = np.random.default_rng(j)
            for i in range(Xp.shape[1]):
                rng.shuffle(Xp[:, i, j])
            scores.append(mae(y, model.predict(Xp, verbose=0).ravel()))
        imp[feat[j]] = float(np.mean(scores) - base)
    return imp

imp = perm_imp(m, Xte, np.log1p(Yte))
pd.Series(imp).sort_values(ascending=False).round(4)

## 8. Latihan Mini

1. Tambahkan fitur regional sintetik (misal `era5_tp` berkorelasi dengan hujan) — lihat efek pada importance & MAE.
2. Bangun klasifikasi multi-kelas kategori (0/1/2) dengan `sparse_categorical_crossentropy`; buat crosstab.
3. Hitung CSI/POD/FAR per kategori dan bandingkan dengan Bab 9 Tabel 9.5.
4. Uji window `w ∈ {7, 14, 30}` di walk-forward 3 blok.
5. Ganti data dengan nyata (BMKG + ERA5 + indeks MJO/ENSO) dan jalankan ulang.